<a href="https://colab.research.google.com/github/HariniRJ/bio-lab/blob/main/lab_pro_6_parallel_cellular_algo_%2C%2C%2Chospital.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import random

# --------------------------------
# HOSPITAL RESOURCE DETAILS
# --------------------------------

patients = [
    {"id": 1, "priority": 10},
    {"id": 2, "priority": 8},
    {"id": 3, "priority": 9},
    {"id": 4, "priority": 5},
    {"id": 5, "priority": 7},
    {"id": 6, "priority": 4},
    {"id": 7, "priority": 6},
    {"id": 8, "priority": 3}
]

NUM_BEDS = 3
NUM_DOCTORS = 2

GRID_SIZE = 3
MAX_ITERATIONS = 10
ALPHA = 0.5

# Each cell stores a patient-priority ordering.
# Earlier patients in the list get considered first.


# --------------------------------
# CREATE INITIAL POPULATION
# --------------------------------

def initialize_population():
    population = []

    for i in range(GRID_SIZE * GRID_SIZE):
        order = [p["id"] for p in patients]
        random.shuffle(order)
        population.append(order)

    return population


# --------------------------------
# EVALUATE FITNESS
# --------------------------------

def evaluate_fitness(order):

    # Consider patients in the given order.
    admitted = order[:NUM_BEDS]
    waiting = order[NUM_BEDS:]

    # Reward admission of higher-priority patients.
    priority_score = sum(
        patients[p - 1]["priority"] for p in admitted
    )

    # Calculate doctor workloads.
    doctor_load = [0] * NUM_DOCTORS

    for i in range(len(admitted)):
        doctor = i % NUM_DOCTORS
        doctor_load[doctor] += 1

    # Penalize unequal workloads.
    workload_difference = max(doctor_load) - min(doctor_load)

    fitness = priority_score - (2 * workload_difference)

    return fitness


# --------------------------------
# FIND BEST NEIGHBOUR
# --------------------------------

def find_best_neighbor(population, index):

    row = index // GRID_SIZE
    col = index % GRID_SIZE

    neighbors = []

    # Von Neumann neighbourhood:
    # Up, down, left and right.

    positions = [
        (row - 1, col),
        (row + 1, col),
        (row, col - 1),
        (row, col + 1)
    ]

    for r, c in positions:

        if 0 <= r < GRID_SIZE and 0 <= c < GRID_SIZE:
            neighbor_index = r * GRID_SIZE + c
            neighbors.append(population[neighbor_index])

    # Select neighbour with highest fitness.
    best_neighbor = max(
        neighbors,
        key=evaluate_fitness
    )

    return best_neighbor


# --------------------------------
# UPDATE CELL STATE
# --------------------------------

def update_cell(current, best_neighbor):

    new_order = current.copy()

    # Move some positions towards the
    # better neighbour's ordering.

    for i in range(len(new_order)):

        if random.random() < ALPHA:

            target_patient = best_neighbor[i]

            target_position = new_order.index(target_patient)

            # Swap positions.
            new_order[i], new_order[target_position] = (
                new_order[target_position],
                new_order[i]
            )

    # Small random change to maintain diversity.
    i, j = random.sample(range(len(new_order)), 2)

    new_order[i], new_order[j] = (
        new_order[j],
        new_order[i]
    )

    return new_order


# --------------------------------
# MAIN PCA ALGORITHM
# --------------------------------

def parallel_cellular_algorithm():

    population = initialize_population()

    best_solution = None
    best_fitness = float("-inf")

    for iteration in range(MAX_ITERATIONS):

        # Evaluate all cells.
        for cell in population:

            fitness = evaluate_fitness(cell)

            if fitness > best_fitness:
                best_fitness = fitness
                best_solution = cell.copy()

        # Use the same old population to update
        # every cell in this iteration.
        new_population = []

        for i in range(len(population)):

            current = population[i]

            best_neighbor = find_best_neighbor(
                population, i
            )

            updated = update_cell(
                current, best_neighbor
            )

            new_population.append(updated)

        population = new_population

        print(
            "Iteration:", iteration + 1,
            "| Best Fitness:", best_fitness
        )

    return best_solution, best_fitness


# --------------------------------
# DISPLAY FINAL ALLOCATION
# --------------------------------

def display_result(solution, fitness):

    admitted = solution[:NUM_BEDS]
    waiting = solution[NUM_BEDS:]

    print("\n========== HOSPITAL RESOURCE ALLOCATION ==========")

    doctor_load = [0] * NUM_DOCTORS

    for i, patient_id in enumerate(admitted):

        doctor = (i % NUM_DOCTORS) + 1
        bed = i + 1

        doctor_load[doctor - 1] += 1

        print(
            f"Patient {patient_id} -> "
            f"Bed {bed}, Doctor {doctor}"
        )

    print("\nPatients waiting:")

    for patient_id in waiting:
        print("Patient", patient_id)

    print("\nDoctor workloads:")

    for i, load in enumerate(doctor_load):
        print(f"Doctor {i + 1}: {load} patients")

    print("\nBest Fitness Score:", fitness)


# --------------------------------
# PROGRAM STARTS HERE
# --------------------------------

best_solution, best_fitness = parallel_cellular_algorithm()

display_result(best_solution, best_fitness)


Iteration: 1 | Best Fitness: 21
Iteration: 2 | Best Fitness: 23
Iteration: 3 | Best Fitness: 23
Iteration: 4 | Best Fitness: 23
Iteration: 5 | Best Fitness: 23
Iteration: 6 | Best Fitness: 23
Iteration: 7 | Best Fitness: 23
Iteration: 8 | Best Fitness: 23
Iteration: 9 | Best Fitness: 23
Iteration: 10 | Best Fitness: 23

========== HOSPITAL RESOURCE ALLOCATION ==========
Patient 7 -> Bed 1, Doctor 1
Patient 3 -> Bed 2, Doctor 2
Patient 1 -> Bed 3, Doctor 1

Patients waiting:
Patient 2
Patient 8
Patient 5
Patient 6
Patient 4

Doctor workloads:
Doctor 1: 2 patients
Doctor 2: 1 patients

Best Fitness Score: 23
